# NB2 – ES Tồn kho: khảo sát, tạo nhãn rủi ro (chưa gán lớp) và kiểm chứng nhãn proxy
**DS317 – Lab 3 | Nhóm 5 (DS317.R11)** · Đề tài: dự báo doanh số và cảnh báo rủi ro tồn kho theo SKU–tuần (Electronics Sales)

| Phần | Nội dung | Yêu cầu task |
|---|---|---|
| 0 | Cấu hình chung (giống NB1) + nạp dataset + nạp `es_sales_wk` từ NB1 | (1) |
| 1 | Khảo sát tồn kho: loại movement và dấu số lượng, tái lập tồn kho cuối tuần, chất lượng 6 chiều, SKU khớp bán hàng, cửa sổ đồng bộ, Hình 7 và các biểu đồ | (5) |
| 2 | Tạo nhãn: `stock_end`, `stock_delta`, `cov_past`, `cov_fwd`, `trend4`, `n_fwd_weeks` (**không gán lớp Low/Medium/High**) | (6) |
| 3 | Kiểm chứng nhãn proxy: theo nhóm sản phẩm, độ nhạy ngưỡng, giảm giá thực tế 4 tuần sau | (7) |
| 4 | Kiểm tra khóa → ghi file xuất | (1) |

**Cần chạy NB1 trước** (NB2 nạp `es_sales_wk` để lấy `qty`, **không tính lại doanh số**).
**File xuất vào `interim/`:** `es_stock_wk`, `es_quality_inventory`
**Cách chạy (Colab):** Runtime → Run all; nếu CSV đã được NB1 giải nén vào `DS317_Nhom5/raw/` (Drive) thì không cần chọn lại zip.

## 0. Cấu hình chung, nạp dataset và nạp kết quả NB1

In [ ]:
# ============================ Ô CẤU HÌNH CHUNG (giữ GIỐNG HỆT ở mọi notebook của nhóm) ============================
import os, sys, re, json, math, time, random, zipfile, shutil, warnings, datetime as dt
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib.dates as mdates
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60); pd.set_option('display.width', 200); pd.set_option('display.max_colwidth', 90)
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3,
                     'axes.spines.top': False, 'axes.spines.right': False})

# ---------------- Hằng số (khai báo MỘT LẦN) ----------------
SEED              = 42
HORIZON_WEEKS     = 4                 # nhìn trước 4 tuần
EPS               = 1e-6              # chặn mẫu số của Coverage
SYNC_START        = '2023-07-03'      # A2: bán hàng và tồn kho cùng bắt đầu 01/07/2023 (thứ Bảy) -> tuần ĐỦ 7 ngày đầu tiên
SYNC_END          = '2025-12-22'      # A2: dữ liệu kết thúc 31/12/2025 (thứ Tư) -> tuần ĐỦ 7 ngày cuối cùng (tuần 29/12 chỉ có 3 ngày, bị loại)
MIN_HISTORY_WEEKS = 26                # SKU phải có >= N tuần trong TRAIN (áp dụng ở notebook chung)
REF_DATE          = None              # ngày lớn nhất trong dữ liệu -> được tính ngay sau khi đọc dữ liệu (xem set_ref_date)
KEY               = ['sku_id', 'week_start']
ACCESS_DATE_ES    = '🟦 dd/mm/yyyy'   # A1: ngày truy cập Kaggle (điền cho khớp Mục 1 của notebook chung)

# ---------------- Đường dẫn ----------------
USE_DRIVE = True       # True: lưu raw/interim trên Google Drive (để NB1 -> NB2 -> notebook chung dùng chung file). False: lưu tạm trong /content
IN_COLAB  = 'google.colab' in sys.modules
BASE = Path('.')
if IN_COLAB:
    BASE = Path('/content')
    if USE_DRIVE:
        try:
            from google.colab import drive
            drive.mount('/content/drive'); BASE = Path('/content/drive/MyDrive')
        except Exception as e:
            print('⚠️ Không mount được Drive, dùng /content (file sẽ mất khi hết phiên):', e)
ROOT = BASE / 'DS317_Nhom5'
DIRS = {k: ROOT / k for k in ['raw', 'interim', 'processed', 'results', 'figures', 'excel']}
for d in DIRS.values(): d.mkdir(parents=True, exist_ok=True)

RAW_TABLES = ['BrandSubcategoryMap', 'Brands', 'Categories', 'CustomerReturns', 'Customers', 'Governorates',
              'InventoryBalances', 'InventoryMovements', 'Locations', 'PaymentMethods', 'Payments', 'ProductVariants',
              'Products', 'Promotions', 'PurchaseOrderLines', 'PurchaseOrders', 'PurchaseReceiptLines', 'PurchaseReceipts',
              'PurchaseReturns', 'SKUSuppliers', 'SalesChannels', 'SalesOrderLines', 'SalesOrders', 'Shipments',
              'ShippingProviders', 'Subcategories', 'Suppliers']          # 27 bảng của Electronics Sales

# ---------------- Hàm dùng chung ----------------
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
set_seed()

def to_snake(name):                                   # SalesOrderLines -> sales_order_lines ; SKUSuppliers -> sku_suppliers
    return re.sub(r'(?<=[a-z0-9])(?=[A-Z])|(?<=[A-Z])(?=[A-Z][a-z])', '_', name).lower()

def week_start_of(s):                                 # ĐỊNH NGHĨA TUẦN DUY NHẤT: thứ Hai 00:00
    s = pd.to_datetime(s)
    return (s - pd.to_timedelta(s.dt.weekday, unit='D')).dt.normalize().astype('datetime64[ns]')

def is_valid_order(df, status_col='OrderStatus', bad=('Cancelled', 'Canceled', 'Failed')):
    # A2: OrderStatus gồm Completed/Complete (hợp lệ) và Cancelled/Canceled (hủy) -> loại các đơn hủy
    return ~df[status_col].isin(bad)

def robust_range(dates, min_share=0.05):
    """Khoảng ngày 'dày đặc': bỏ các ngày lẻ tẻ (số bản ghi < 5% trung vị số bản ghi/ngày) coi như ngoại lai thời gian."""
    d = pd.to_datetime(dates, errors='coerce').dropna().dt.normalize()
    if d.empty: return pd.NaT, pd.NaT
    c = d.value_counts(); keep = c[c >= max(1, min_share * c.median())].index
    return keep.min(), keep.max()

def set_ref_date(movements_raw):                      # REF_DATE = ngày lớn nhất (dày đặc) của nhật ký tồn kho
    global REF_DATE
    REF_DATE = robust_range(movements_raw['MovementDateTime'])[1]
    return REF_DATE

def read_tables(names, verbose=True):                 # đọc bảng thành es_<tên_bảng_snake>_raw
    out = {}
    for t in names:
        df = pd.read_csv(DIRS['raw'] / f'{t}.csv', low_memory=False)
        out[t] = df; globals()[f'es_{to_snake(t)}_raw'] = df
        if verbose: print(f'  es_{to_snake(t)}_raw'.ljust(36), f'{df.shape[0]:>9,} dòng × {df.shape[1]:>2} cột')
    return out

def prepare_raw():
    """Chỉ cần đưa file zip dataset vào Colab (kéo thả vào /content, hoặc để trong Drive, hoặc chọn file khi hộp thoại hiện ra)."""
    raw = DIRS['raw']; miss = [t for t in RAW_TABLES if not (raw / f'{t}.csv').exists()]
    if not miss: print(f'✅ Đã có đủ {len(RAW_TABLES)} bảng trong {raw}'); return
    def find_zip():
        cands = []
        for b in [Path('/content'), Path('.'), ROOT, raw, BASE]:
            if b.exists(): cands += sorted(b.glob('*.zip'))
        for c in cands:
            try:
                with zipfile.ZipFile(c) as z:
                    if any(n.endswith('SalesOrderLines.csv') for n in z.namelist()): return c
            except Exception: pass
        return None
    z_path = find_zip()
    if z_path is None and IN_COLAB:
        print('📦 Chưa thấy file zip dataset – hãy chọn file archive.zip (Electronics Sales) ở hộp thoại bên dưới...')
        from google.colab import files
        files.upload(); z_path = find_zip()
    if z_path is None:
        raise FileNotFoundError('Không tìm thấy file zip chứa SalesOrderLines.csv. Hãy đặt archive.zip cạnh notebook (hoặc /content, hoặc ' + str(ROOT) + ').')
    print('📦 Giải nén', z_path, '->', raw)
    with zipfile.ZipFile(z_path) as z:
        for m in z.namelist():
            if m.lower().endswith('.csv'):
                with z.open(m) as src, open(raw / Path(m).name, 'wb') as dst: shutil.copyfileobj(src, dst)
    miss = [t for t in RAW_TABLES if not (raw / f'{t}.csv').exists()]
    assert not miss, f'Thiếu bảng sau khi giải nén: {miss}'
    print(f'✅ Đã có đủ {len(RAW_TABLES)} bảng')

def save_df(df, name, folder='interim'):
    p = DIRS[folder] / name
    try:
        df.to_parquet(p.with_suffix('.parquet'), index=False)
        p.with_suffix('.csv').unlink(missing_ok=True); return p.with_suffix('.parquet')
    except Exception:
        df.to_csv(p.with_suffix('.csv'), index=False); p.with_suffix('.parquet').unlink(missing_ok=True); return p.with_suffix('.csv')

def load_df(name, folder='interim'):
    for ext in ('.parquet', '.csv'):
        p = (DIRS[folder] / name).with_suffix(ext)
        if p.exists():
            df = pd.read_parquet(p) if ext == '.parquet' else pd.read_csv(p)
            if 'week_start' in df.columns: df['week_start'] = pd.to_datetime(df['week_start'])
            return df
    return None

STEP_LOG = []
def log_step(df, step):
    STEP_LOG.append({'bước': step, 'số_dòng': len(df), 'số_SKU': df.sku_id.nunique(), 'số_tuần': df.week_start.nunique()})
    print(f'  [{step}] dòng={len(df):,} SKU={df.sku_id.nunique():,} tuần={df.week_start.nunique()}')

def save_fig(fig, name):
    p = DIRS['figures'] / f'fig_{name}.png'; fig.savefig(p, dpi=200, bbox_inches='tight'); print('đã lưu', p.name)

# Quy tắc dấu số lượng của InventoryMovements (A2, kiểm chứng khớp 100% InventoryBalances ở NB2):
#   Quantity luôn dương; tác động lên tồn KHẢ DỤNG = +Quantity nếu ToCondition=='Available', -Quantity nếu FromCondition=='Available'.
#   (Internal Transfer: từ Available -> Available nên triệt tiêu ở mức toàn chuỗi; Non-Sellable không tính vào tồn khả dụng)
def movement_delta(im):
    return im['Quantity'] * (im['ToCondition'].eq('Available').astype(int) - im['FromCondition'].eq('Available').astype(int))

def stock_end_chain_weekly(im, last_week=None):
    """Tồn kho khả dụng CUỐI TUẦN (toàn chuỗi) theo SKU-tuần = cộng dồn delta từ đầu dữ liệu. Trả về [sku_id, week_start, stock_end, delta_wk]."""
    last_week = pd.Timestamp(SYNC_END if last_week is None else last_week)
    d = pd.DataFrame({'sku_id': im['SKU_ID'].astype(str).values,
                      'week_start': week_start_of(im['MovementDateTime']).values,
                      'delta': movement_delta(im).values})
    d = d[d['week_start'] <= last_week]
    w = d.groupby(['sku_id', 'week_start'])['delta'].sum().unstack('week_start', fill_value=0)
    w = w.reindex(columns=pd.date_range(w.columns.min(), last_week, freq='7D'), fill_value=0); w.columns.name = 'week_start'
    out = w.cumsum(axis=1).stack().rename('stock_end').reset_index()
    out['delta_wk'] = w.stack().values
    return out

def check_and_save(df, name, key=None, folder='interim'):
    """Cell cuối mỗi notebook: kiểm tra khóa không trùng -> in số dòng, tuần min–max, số thiếu -> rồi mới ghi file."""
    print(f'──────── {name} ────────')
    if key is not None:
        dup = int(df.duplicated(key).sum()); assert dup == 0, f'{name}: có {dup} khóa {key} trùng'
        print(f'khóa {key} trùng: {dup}  ✅')
    print(f'số dòng: {len(df):,} | số cột: {df.shape[1]}')
    if 'week_start' in df.columns:
        df['week_start'] = pd.to_datetime(df['week_start']).astype('datetime64[ns]')
        print(f'tuần min–max: {df.week_start.min().date()} → {df.week_start.max().date()} | số tuần: {df.week_start.nunique()}')
    if 'sku_id' in df.columns: print(f'số SKU: {df.sku_id.nunique():,}')
    miss = df.isna().sum(); miss = miss[miss > 0]
    print('số giá trị thiếu theo cột:', miss.to_dict() if len(miss) else 'không có')
    assert 'CustomerName' not in df.columns, 'Không được xuất cột CustomerName (riêng tư)'
    p = save_df(df, name, folder); print('→ đã ghi', p, '\n'); return p

print('Cấu hình xong | ROOT =', ROOT.resolve(), '| SYNC:', SYNC_START, '→', SYNC_END)

In [ ]:
prepare_raw()
es_sales_wk = load_df('es_sales_wk')
if es_sales_wk is None:
    raise FileNotFoundError('Chưa có es_sales_wk trong ' + str(DIRS['interim']) + ' → hãy chạy NB1 (ES – Bán hàng) trước.')
assert es_sales_wk.week_start.min() >= pd.Timestamp(SYNC_START) and es_sales_wk.week_start.max() <= pd.Timestamp(SYNC_END), 'es_sales_wk lệch cửa sổ SYNC – chạy lại NB1 cùng cấu hình'
print(f'es_sales_wk (từ NB1): {es_sales_wk.shape[0]:,} dòng | {es_sales_wk.sku_id.nunique()} SKU | tuần {es_sales_wk.week_start.min().date()} → {es_sales_wk.week_start.max().date()}')

## 1. Khảo sát tồn kho *(A2)*
### 1.1 Đọc `InventoryMovements`, `PurchaseOrders`, `Products` và các bảng đối chiếu

In [ ]:
t0 = time.time()
TABLES_NB2 = ['InventoryMovements', 'PurchaseOrders', 'Products', 'InventoryBalances', 'ProductVariants', 'Subcategories', 'Categories', 'Brands',
              'PurchaseReceipts', 'PurchaseReceiptLines', 'PurchaseOrderLines', 'PurchaseReturns', 'CustomerReturns', 'Locations', 'SalesOrders', 'SalesOrderLines']
RAW = read_tables(TABLES_NB2)
set_ref_date(es_inventory_movements_raw)
im = es_inventory_movements_raw.copy(); im['MovementDateTime'] = pd.to_datetime(im['MovementDateTime'])
print(f'\nĐọc xong trong {time.time() - t0:.1f}s | REF_DATE = {REF_DATE.date()}')
print('Thời gian nhật ký tồn kho:', im.MovementDateTime.min(), '→', im.MovementDateTime.max(), '| số SKU:', im.SKU_ID.nunique(), '| số kho/cửa hàng:', pd.concat([im.FromLocationID, im.ToLocationID]).nunique())

### 1.2 Loại movement và dấu số lượng
`Quantity` trong nhật ký **luôn dương**; chiều tác động lên tồn kho được suy ra từ `FromLocation/ToLocation` và `FromCondition/ToCondition`.
Quy tắc (đã đưa vào hàm dùng chung `movement_delta`): **tác động lên tồn khả dụng = +Quantity nếu `ToCondition = Available`, −Quantity nếu `FromCondition = Available`**.

In [ ]:
im['delta'] = movement_delta(im)
print('Quantity: min =', im.Quantity.min(), '| max =', im.Quantity.max(), '| số dòng Quantity <= 0:', int((im.Quantity <= 0).sum()))
typ = im.groupby('MovementType').agg(số_dòng=('Quantity', 'size'), tổng_số_lượng=('Quantity', 'sum'), Qty_min=('Quantity', 'min'), Qty_max=('Quantity', 'max'), Qty_TB=('Quantity', 'mean'),
                                      có_From=('FromLocationID', lambda s: s.notna().mean()), có_To=('ToLocationID', lambda s: s.notna().mean()),
                                      tác_động_ròng_lên_tồn_khả_dụng=('delta', 'sum'), ngày_đầu=('MovementDateTime', 'min'), ngày_cuối=('MovementDateTime', 'max')).sort_values('số_dòng', ascending=False)
typ['dấu'] = np.select([typ.tác_động_ròng_lên_tồn_khả_dụng > 0, typ.tác_động_ròng_lên_tồn_khả_dụng < 0], ['+ (nhập)', '− (xuất)'], default='0 (nội bộ, triệt tiêu)')
display(typ.round(3))
combo = im.groupby(['MovementType', im.FromCondition.fillna('—').rename('FromCondition'), im.ToCondition.fillna('—').rename('ToCondition')]).agg(số_dòng=('Quantity', 'size'), tổng_Quantity=('Quantity', 'sum'), tác_động_lên_tồn_khả_dụng=('delta', 'sum'))
display(combo)

#### Kiểm chứng quy tắc dấu bằng `InventoryBalances` (tồn cuối kỳ theo SKU–địa điểm)
Nếu quy tắc đúng thì cộng dồn toàn bộ nhật ký phải **trùng khớp** `AvailableQuantity` (và `NonSellableQuantity`) ở cuối dữ liệu.

In [ ]:
def end_balance(col_cond, qty_sign_to='Available'):
    a = im[im.ToCondition.eq(col_cond)].groupby(['SKU_ID', 'ToLocationID'])['Quantity'].sum().rename('in'); a.index.names = ['SKU_ID', 'LocationID']
    b = im[im.FromCondition.eq(col_cond)].groupby(['SKU_ID', 'FromLocationID'])['Quantity'].sum().rename('out'); b.index.names = ['SKU_ID', 'LocationID']
    x = pd.concat([a, b], axis=1).fillna(0); return (x['in'] - x['out']).rename('rebuilt')
ib = es_inventory_balances_raw.set_index(['SKU_ID', 'LocationID'])
rec = pd.DataFrame({'rebuilt_avail': end_balance('Available'), 'rebuilt_nonsell': end_balance('Non-Sellable')}).join(ib[['AvailableQuantity', 'NonSellableQuantity']], how='outer').fillna(0)
m_av = (rec.rebuilt_avail == rec.AvailableQuantity).mean(); m_ns = (rec.rebuilt_nonsell == rec.NonSellableQuantity).mean()
print(f'Số cặp SKU–địa điểm đối chiếu: {len(rec):,}')
print(f'Khớp AvailableQuantity: {m_av * 100:.2f}% | khớp NonSellableQuantity: {m_ns * 100:.2f}% | chênh lệch tuyệt đối lớn nhất (Available): {int((rec.rebuilt_avail - rec.AvailableQuantity).abs().max())}')
print('✅ Quy tắc dấu số lượng được xác nhận' if m_av == 1 else '⚠️ Có lệch – kiểm tra lại quy tắc dấu')

### 1.3 Tái lập tồn kho cuối tuần bằng cộng dồn và kiểm tra không âm
`stock_end(t)` = tổng delta của mọi movement có thời điểm **trước hết chủ nhật của tuần t** (cộng dồn từ `Opening Balance` 01/07/2023). Kiểm tra không âm ở 3 mức: toàn chuỗi (SKU–tuần), từng địa điểm (cuối tuần) và từng sự kiện.

In [ ]:
stock_long = stock_end_chain_weekly(im)                       # [sku_id, week_start, stock_end, delta_wk], từ tuần 26/06/2023 đến SYNC_END
print('Số SKU–tuần tái lập:', f'{len(stock_long):,}', '| tuần', stock_long.week_start.min().date(), '→', stock_long.week_start.max().date())
neg_chain = int((stock_long.stock_end < 0).sum())

# mức địa điểm: sự kiện vào (To, Available) và ra (From, Available)
ev = pd.concat([
    pd.DataFrame({'sku': im.SKU_ID, 'loc': im.ToLocationID, 'dt': im.MovementDateTime, 'delta': im.Quantity.where(im.ToCondition.eq('Available'), 0), 'ord': 0}),
    pd.DataFrame({'sku': im.SKU_ID, 'loc': im.FromLocationID, 'dt': im.MovementDateTime, 'delta': -im.Quantity.where(im.FromCondition.eq('Available'), 0), 'ord': 1})])
ev = ev[ev['loc'].notna() & (ev['delta'] != 0)].sort_values(['sku', 'loc', 'dt', 'ord'])
ev['cum'] = ev.groupby(['sku', 'loc'])['delta'].cumsum()
ev['week_start'] = week_start_of(ev['dt'])
neg_event = int((ev.cum < 0).sum())
loc_week = ev.groupby(['sku', 'loc', 'week_start'])['cum'].last()
neg_loc_week = int((loc_week < 0).sum())
chk = pd.DataFrame([{'mức_kiểm_tra': 'toàn chuỗi – SKU–tuần (cuối tuần)', 'số_điểm': len(stock_long), 'số_điểm_âm': neg_chain, 'min': int(stock_long.stock_end.min())},
                    {'mức_kiểm_tra': 'từng địa điểm – SKU–địa điểm–tuần (cuối tuần)', 'số_điểm': len(loc_week), 'số_điểm_âm': neg_loc_week, 'min': int(loc_week.min())},
                    {'mức_kiểm_tra': 'từng địa điểm – từng sự kiện (nhập trước xuất khi trùng giờ)', 'số_điểm': len(ev), 'số_điểm_âm': neg_event, 'min': int(ev.cum.min())}])
display(chk)
assert neg_chain == 0, 'Tồn kho toàn chuỗi âm – kiểm tra quy tắc dấu'
print('✅ Tồn kho tái lập KHÔNG ÂM ở mức SKU–tuần', '(và ở mức địa điểm)' if neg_loc_week == 0 and neg_event == 0 else '(có điểm âm ở mức địa điểm/sự kiện – ghi chú ở bảng chất lượng)')
del ev, loc_week

### 1.4 Chất lượng dữ liệu tồn kho theo 6 chiều → xuất `es_quality_inventory`

In [ ]:
D_COMP, D_UNIQ, D_VALID, D_CONS, D_ACC, D_TIME = ('1. Đầy đủ', '2. Duy nhất', '3. Hợp lệ', '4. Nhất quán', '5. Chính xác / ngoại lai', '6. Kịp thời / phạm vi thời gian')
QUALITY = []
def add_q(dim, table, check, n, total, note=''):
    QUALITY.append({'dimension': dim, 'table': table, 'check': check, 'n_flagged': int(n), 'n_total': int(total), 'pct': round(100 * float(n) / max(int(total), 1), 3), 'note': note})
N = len(im)
# 1. ĐẦY ĐỦ: cột thiếu theo cấu trúc (thiếu có chủ đích) và thiếu bất thường
for c in ['InventoryMovementID', 'MovementDateTime', 'SKU_ID', 'MovementType', 'Quantity', 'UnitCostAtMovement']: add_q(D_COMP, 'InventoryMovements', f'thiếu {c} (cột bắt buộc)', im[c].isna().sum(), N)
bad_ft = im.MovementType.isin(['Sale', 'Replacement Issue', 'Purchase Return', 'Internal Transfer']) & im.FromLocationID.isna()
bad_tt = im.MovementType.isin(['Opening Balance', 'Purchase Receipt', 'Internal Transfer']) & im.ToLocationID.isna()
add_q(D_COMP, 'InventoryMovements', 'loại xuất/chuyển thiếu FromLocationID', bad_ft.sum(), N); add_q(D_COMP, 'InventoryMovements', 'loại nhập/chuyển thiếu ToLocationID', bad_tt.sum(), N)
for c, t_ in [('PurchaseReceiptLineID', 'Purchase Receipt'), ('SalesOrderLineID', 'Sale'), ('CustomerReturnID', 'Customer Return'), ('PurchaseReturnID', 'Purchase Return')]:
    add_q(D_COMP, 'InventoryMovements', f'{t_} thiếu {c} (khóa liên kết chứng từ)', (im.MovementType.eq(t_) & im[c].isna()).sum(), int(im.MovementType.eq(t_).sum()))
add_q(D_COMP, 'InventoryMovements', 'Adjustment thiếu AdjustmentReason', (im.MovementType.eq('Adjustment') & im.AdjustmentReason.isna()).sum(), int(im.MovementType.eq('Adjustment').sum()), 'thiếu lý do điều chỉnh – chỉ ảnh hưởng giải thích, không ảnh hưởng số lượng')
# 2. DUY NHẤT
add_q(D_UNIQ, 'InventoryMovements', 'InventoryMovementID trùng', im.InventoryMovementID.duplicated().sum(), N)
add_q(D_UNIQ, 'InventoryMovements', 'dòng trùng nội dung (bỏ qua ID)', im.drop(columns='InventoryMovementID').duplicated().sum(), N, 'có thể hợp lệ (nhiều giao dịch giống nhau cùng giờ); không xóa')
add_q(D_UNIQ, 'PurchaseOrders', 'PurchaseOrderID trùng', es_purchase_orders_raw.PurchaseOrderID.duplicated().sum(), len(es_purchase_orders_raw))
add_q(D_UNIQ, 'Products', 'ProductID trùng', es_products_raw.ProductID.duplicated().sum(), len(es_products_raw))
# 3. HỢP LỆ
add_q(D_VALID, 'InventoryMovements', 'Quantity <= 0', (im.Quantity <= 0).sum(), N)
add_q(D_VALID, 'InventoryMovements', 'MovementType ngoài 8 loại đã biết', (~im.MovementType.isin(typ.index)).sum(), N)
cond_ok = {'Available', 'Non-Sellable'}
add_q(D_VALID, 'InventoryMovements', 'From/ToCondition ngoài {Available, Non-Sellable}', ((im.FromCondition.notna() & ~im.FromCondition.isin(cond_ok)) | (im.ToCondition.notna() & ~im.ToCondition.isin(cond_ok))).sum(), N)
add_q(D_VALID, 'InventoryMovements', 'UnitCostAtMovement <= 0', (im.UnitCostAtMovement <= 0).sum(), N)
add_q(D_VALID, 'InventoryMovements', 'MovementDateTime không đọc được', im.MovementDateTime.isna().sum(), N)
add_q(D_VALID, 'PurchaseOrders', 'PurchaseOrderStatus viết khác chuẩn (Close vs Closed)', es_purchase_orders_raw.PurchaseOrderStatus.eq('Close').sum(), len(es_purchase_orders_raw), 'cùng nghĩa với Closed – chuẩn hóa nếu dùng')
add_q(D_VALID, 'es_stock_wk (tái lập)', 'tồn kho cuối tuần âm (toàn chuỗi)', neg_chain, len(stock_long), 'kỳ vọng 0')
# 4. NHẤT QUÁN
so_u = es_sales_orders_raw.drop_duplicates(); so_u = so_u.assign(OrderDate=pd.to_datetime(so_u.OrderDate))
sol = es_sales_order_lines_raw.merge(so_u[['SalesOrderID', 'OrderStatus']], on='SalesOrderID', how='left')
valid_ids = set(sol.loc[is_valid_order(sol), 'SalesOrderLineID']); cancel_ids = set(sol.loc[~is_valid_order(sol), 'SalesOrderLineID'])
sale_mv = im[im.MovementType.eq('Sale')]
add_q(D_CONS, 'InventoryMovements ↔ SalesOrderLines', 'movement Sale không trỏ tới dòng đơn HỢP LỆ', (~sale_mv.SalesOrderLineID.isin(valid_ids)).sum(), len(sale_mv), 'kỳ vọng 0: nhật ký chỉ ghi xuất kho cho đơn hợp lệ')
add_q(D_CONS, 'SalesOrderLines ↔ InventoryMovements', 'dòng đơn hợp lệ KHÔNG có movement Sale', (~pd.Series(list(valid_ids)).isin(sale_mv.SalesOrderLineID)).sum(), len(valid_ids))
add_q(D_CONS, 'InventoryMovements ↔ SalesOrderLines', 'movement Sale trỏ tới dòng của đơn hủy', sale_mv.SalesOrderLineID.isin(cancel_ids).sum(), len(sale_mv))
_q = sale_mv.groupby('SalesOrderLineID')['Quantity'].sum().rename('mv_qty').reset_index().merge(es_sales_order_lines_raw[['SalesOrderLineID', 'Quantity']], on='SalesOrderLineID')
add_q(D_CONS, 'InventoryMovements ↔ SalesOrderLines', 'số lượng xuất kho ≠ Quantity của dòng đơn', (_q.mv_qty != _q.Quantity).sum(), len(_q))
_pr = im[im.MovementType.eq('Purchase Receipt')].merge(es_purchase_receipt_lines_raw[['PurchaseReceiptLineID', 'AcceptedQuantity']], on='PurchaseReceiptLineID', how='left')
add_q(D_CONS, 'InventoryMovements ↔ PurchaseReceiptLines', 'Purchase Receipt: số lượng ≠ AcceptedQuantity', (_pr.Quantity != _pr.AcceptedQuantity).sum(), len(_pr))
add_q(D_CONS, 'InventoryMovements ↔ CustomerReturns', 'Customer Return trỏ tới CustomerReturnID không tồn tại', (im.MovementType.eq('Customer Return') & ~im.CustomerReturnID.isin(es_customer_returns_raw.CustomerReturnID)).sum(), int(im.MovementType.eq('Customer Return').sum()))
add_q(D_CONS, 'InventoryMovements ↔ InventoryBalances', 'cặp SKU–địa điểm: tồn tái lập ≠ AvailableQuantity cuối kỳ', (rec.rebuilt_avail != rec.AvailableQuantity).sum(), len(rec), 'kỳ vọng 0 – xác nhận quy tắc dấu')
add_q(D_CONS, 'InventoryMovements ↔ ProductVariants', 'SKU không có trong ProductVariants', (~im.SKU_ID.isin(es_product_variants_raw.SKU_ID)).sum(), N)
_rc = es_purchase_receipts_raw.merge(es_purchase_orders_raw[['PurchaseOrderID', 'OrderDate']], on='PurchaseOrderID', how='left')
add_q(D_CONS, 'PurchaseOrders ↔ PurchaseReceipts', 'ReceiptDate trước OrderDate của đơn mua', (pd.to_datetime(_rc.ReceiptDate) < pd.to_datetime(_rc.OrderDate)).sum(), len(_rc))
# 5. CHÍNH XÁC / NGOẠI LAI
for t_, g in im.groupby('MovementType'):
    q1, q3 = g.Quantity.quantile([.25, .75]); add_q(D_ACC, 'InventoryMovements', f'{t_}: Quantity > Q3 + 3·IQR', (g.Quantity > q3 + 3 * (q3 - q1)).sum(), len(g), f'max = {g.Quantity.max()}')
_uc = im.groupby('SKU_ID')['UnitCostAtMovement'].agg(['min', 'max']); add_q(D_ACC, 'InventoryMovements', 'SKU có giá vốn max/min > 3×', ((_uc['max'] / _uc['min']) > 3).sum(), len(_uc), 'biến động giá vốn theo thời gian / nhà cung cấp')
add_q(D_ACC, 'es_stock_wk (tái lập)', 'stock_end SKU–tuần > P99 của chính nó + 3·IQR', (stock_long.stock_end > stock_long.groupby('sku_id').stock_end.transform(lambda s: s.quantile(.75) + 3 * (s.quantile(.75) - s.quantile(.25)))).sum(), len(stock_long), f'stock_end max = {stock_long.stock_end.max()}, trung vị = {stock_long.stock_end.median():.0f}')
# 6. KỊP THỜI
lo_m, hi_m = robust_range(im.MovementDateTime)
add_q(D_TIME, 'InventoryMovements', f'movement ngoài khoảng dày đặc [{lo_m.date()}, {hi_m.date()}]', ((im.MovementDateTime.dt.normalize() < lo_m) | (im.MovementDateTime.dt.normalize() > hi_m)).sum(), N)
add_q(D_TIME, 'InventoryMovements', f'movement thuộc tuần ngoài cửa sổ SYNC (trước {SYNC_START} hoặc sau {(pd.Timestamp(SYNC_END) + pd.Timedelta(days=6)).date()})',
      ((im.MovementDateTime < SYNC_START) | (im.MovementDateTime >= pd.Timestamp(SYNC_END) + pd.Timedelta(days=7))).sum(), N, 'chỉ dùng cho tái lập tồn đầu kỳ (Opening Balance 01/07) – không đưa vào mẫu')
_po = es_purchase_orders_raw; add_q(D_TIME, 'PurchaseOrders', f'ExpectedDeliveryDate sau REF_DATE ({REF_DATE.date()})', (pd.to_datetime(_po.ExpectedDeliveryDate) > REF_DATE).sum(), len(_po), 'ngày giao dự kiến ở tương lai: bình thường với đơn mua cuối kỳ')
add_q(D_TIME, 'Products', 'ActiveFrom sau SYNC_START (SKU ra mắt trong kỳ)', (pd.to_datetime(es_products_raw.ActiveFrom) > SYNC_START).sum(), len(es_products_raw), 'vòng đời sản phẩm: giải thích tuần qty = 0 trước khi ra mắt')
add_q(D_TIME, 'Products', 'ActiveTo trước SYNC_END (SKU ngừng bán trong kỳ)', (pd.to_datetime(es_products_raw.ActiveTo) < SYNC_END).sum(), len(es_products_raw), 'SKU ngừng bán nhưng còn tồn → rủi ro tồn kho thật')
es_quality_inventory = pd.DataFrame(QUALITY); display(es_quality_inventory.drop(columns=['n_total']))

### 1.5 SKU khớp với bán hàng *(kiểm chứng sơ bộ: 871 SKU)*

In [ ]:
sk_inv, sk_sal = set(im.SKU_ID), set(es_sales_wk.sku_id); sk_pv = set(es_product_variants_raw.SKU_ID)
match = pd.DataFrame([{'tập SKU': 'ProductVariants (danh mục SKU)', 'số SKU': len(sk_pv)}, {'tập SKU': 'InventoryMovements (có tồn kho)', 'số SKU': len(sk_inv)},
                      {'tập SKU': 'es_sales_wk (có bán, từ NB1)', 'số SKU': len(sk_sal)}, {'tập SKU': 'KHỚP: tồn kho ∩ bán hàng', 'số SKU': len(sk_inv & sk_sal)},
                      {'tập SKU': 'chỉ có tồn kho, không bán', 'số SKU': len(sk_inv - sk_sal)}, {'tập SKU': 'chỉ có bán, không có tồn kho', 'số SKU': len(sk_sal - sk_inv)},
                      {'tập SKU': 'trong danh mục nhưng không có tồn kho', 'số SKU': len(sk_pv - sk_inv)}])
display(match)
SKU_OK = sorted(sk_inv & sk_sal); print('SKU chỉ có tồn kho (loại khỏi mẫu):', sorted(sk_inv - sk_sal))
print(f'→ {len(SKU_OK)} SKU khớp', '(= 871 ở kiểm chứng sơ bộ ✅)' if len(SKU_OK) == 871 else '(KHÁC 871 ở kiểm chứng sơ bộ ⚠️ – đối chiếu lại Thuyết minh)')
add_q(D_CONS, 'InventoryMovements ↔ es_sales_wk', 'SKU có tồn kho nhưng không có bán (loại khỏi mẫu)', len(sk_inv - sk_sal), len(sk_inv), 'SKU khớp = ' + str(len(SKU_OK)))
es_quality_inventory = pd.DataFrame(QUALITY)

### 1.6 Bảng min/max khẳng định cửa sổ đồng bộ `SYNC_START` – `SYNC_END`
Cửa sổ chung = giao của khoảng ngày **dày đặc** của bán hàng và tồn kho; tuần đầu/cuối phải **đủ 7 ngày**.

In [ ]:
srcs = [('SalesOrders.OrderDate', es_sales_orders_raw['OrderDate'], True), ('InventoryMovements.MovementDateTime', im['MovementDateTime'], True),
        ('PurchaseOrders.OrderDate', es_purchase_orders_raw['OrderDate'], False), ('PurchaseReceipts.ReceiptDate', es_purchase_receipts_raw['ReceiptDate'], False)]
rows = []
for nm, s, core in srcs:
    s = pd.to_datetime(s, errors='coerce'); lo, hi = robust_range(s)
    rows.append({'nguồn': nm, 'min thô': s.min().date(), 'max thô': s.max().date(), 'min dày đặc': lo.date(), 'max dày đặc': hi.date(), 'quyết định cửa sổ?': 'CÓ' if core else 'không (chứng từ phụ)'})
mm = pd.DataFrame(rows); display(mm)
core_lo = max(pd.Timestamp(r['min dày đặc']) for r in rows if r['quyết định cửa sổ?'] == 'CÓ'); core_hi = min(pd.Timestamp(r['max dày đặc']) for r in rows if r['quyết định cửa sổ?'] == 'CÓ')
first_mon = core_lo + pd.Timedelta(days=(7 - core_lo.weekday()) % 7)                       # thứ Hai đầu tiên >= ngày bắt đầu
last_mon = core_hi - pd.Timedelta(days=core_hi.weekday()) - (pd.Timedelta(days=7) if core_hi.weekday() != 6 else pd.Timedelta(0))   # thứ Hai của tuần đủ 7 ngày cuối cùng
sync = pd.DataFrame([{'giao khoảng dày đặc (bán ∩ tồn)': f'{core_lo.date()} ({core_lo.day_name()}) → {core_hi.date()} ({core_hi.day_name()})',
                      'SYNC_START tính được': first_mon.date(), 'SYNC_END tính được': last_mon.date(), 'SYNC_START cấu hình': SYNC_START, 'SYNC_END cấu hình': SYNC_END,
                      'số tuần trong cửa sổ': len(pd.date_range(SYNC_START, SYNC_END, freq='7D'))}])
display(sync)
assert first_mon == pd.Timestamp(SYNC_START) and last_mon == pd.Timestamp(SYNC_END), 'Cửa sổ cấu hình KHÔNG khớp min/max dữ liệu – sửa SYNC_START/SYNC_END ở ô cấu hình (mọi notebook)'
print('✅ SYNC_START / SYNC_END khớp min/max của dữ liệu.  Lưu ý: đồng bộ SYNC_END =', SYNC_END, 'trong ô cấu hình của NOTEBOOK CHUNG (hiện đang để 2025-12-29, tuần chỉ có 3 ngày).')

### 1.7 Biểu đồ khảo sát: **Hình 7**, tồn kho lũy kế vài SKU, phân bố `stock_end`, số movement theo loại

In [ ]:
sl = stock_long[stock_long.sku_id.isin(SKU_OK) & (stock_long.week_start >= SYNC_START)]
w_tot = sl.groupby('week_start')['stock_end'].sum(); w_qty = es_sales_wk.groupby('week_start')['qty'].sum().reindex(w_tot.index).fillna(0)
fig, ax = plt.subplots(figsize=(13, 4.2))
ax.plot(w_tot.index, w_tot.values, c='#4C78A8', lw=2, label='Tồn kho cuối tuần (tổng chuỗi, trục trái)'); ax.set_ylabel('Tồn kho (đơn vị)', color='#4C78A8'); ax.set_ylim(bottom=0)
ax2 = ax.twinx(); ax2.plot(w_qty.index, w_qty.values, c='#F58518', lw=1.5, label='Doanh số tuần (trục phải)'); ax2.set_ylabel('Doanh số / tuần (đơn vị)', color='#F58518'); ax2.set_ylim(bottom=0); ax2.grid(False)
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels(); ax.legend(h1 + h2, l1 + l2, loc='lower right')
ax.set_title(f'Hình 7 – Tồn kho và doanh số theo tuần ({len(SKU_OK)} SKU khớp)'); plt.tight_layout(); plt.show(); save_fig(fig, 'hinh7_ton_kho_va_doanh_so_theo_tuan')
print(f'Tồn kho tổng: TB={w_tot.mean():,.0f}, min={w_tot.min():,}, max={w_tot.max():,} | doanh số/tuần TB={w_qty.mean():,.0f} → tồn ≈ {w_tot.mean() / w_qty.mean():.1f} tuần bán')

In [ ]:
rng_ = np.random.default_rng(SEED); top = es_sales_wk.groupby('sku_id').qty.sum().nlargest(3).index.tolist(); rnd = list(rng_.choice([s for s in SKU_OK if s not in top], 3, replace=False))
fig, ax = plt.subplots(figsize=(13, 4))
for s in top + rnd:
    d = sl[sl.sku_id == s]; ax.plot(d.week_start, d.stock_end, lw=1.3, label=f'{s}' + (' (bán cao)' if s in top else ' (ngẫu nhiên)'))
ax.set_title('Tồn kho lũy kế cuối tuần của một số SKU'); ax.set_ylabel('stock_end'); ax.legend(ncol=3, fontsize=8); plt.tight_layout(); plt.show(); save_fig(fig, 'ton_kho_luy_ke_vai_sku')

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].hist(sl.stock_end, bins=60, color='#4C78A8'); ax[0].axvline(sl.stock_end.median(), c='red', ls='--', label=f'trung vị {sl.stock_end.median():.0f}'); ax[0].axvline(sl.stock_end.quantile(.95), c='darkorange', ls='--', label=f'P95 {sl.stock_end.quantile(.95):.0f}')
ax[0].set_title('Phân bố stock_end (SKU–tuần, toàn chuỗi)'); ax[0].set_xlabel('stock_end'); ax[0].legend()
mt_c = typ.sort_values('số_dòng'); ax[1].barh(mt_c.index, mt_c.số_dòng, color='#54A24B'); ax[1].set_xscale('log')
for i, (n, dau) in enumerate(zip(mt_c.số_dòng, mt_c['dấu'])): ax[1].text(n * 1.1, i, f'{n:,}  [{dau}]', va='center', fontsize=8)
ax[1].set_xlim(right=mt_c.số_dòng.max() * 40); ax[1].set_title('Số movement theo loại (trục log)'); plt.tight_layout(); plt.show(); save_fig(fig, 'phan_bo_stock_end_va_movement')

## 2. Tạo nhãn và đặc trưng tồn kho *(A11)*
Định nghĩa (tại tuần t, mọi đặc trưng chỉ dùng thông tin ≤ t; `cov_fwd` là **nhãn** nhìn tương lai):

| Cột | Định nghĩa |
|---|---|
| `stock_end` | tồn khả dụng cuối tuần t (toàn chuỗi) – cộng dồn movement |
| `stock_delta` | `stock_end(t) − stock_end(t−1)` |
| `cov_past` | `stock_end ÷ max(TB qty 4 tuần trước t−4…t−1, ε)` |
| `cov_fwd` | `stock_end(t) ÷ max(TB qty **thực** t+1…t+4, ε)` – **chỉ tính khi đủ 4 tuần phía sau** |
| `trend4` | hệ số góc hồi quy của `qty` trên 4 tuần t−3…t (≤ t) |
| `n_fwd_weeks` | số tuần phía sau (0–4) còn nằm trong cửa sổ SYNC để tính `cov_fwd` |

`qty` lấy từ `es_sales_wk` (NB1). Tuần nằm ngoài khoảng bán của SKU nhưng trong cửa sổ SYNC được coi `qty = 0` (dữ liệu bán hàng phủ kín cửa sổ nên đây là "không bán" thật). **Không gán lớp Low/Medium/High ở đây** (ngưỡng chỉ tính trên train ở notebook chung).

In [ ]:
# ---- 2.1 stock_end, stock_delta, unit_cost theo SKU–tuần (cộng dồn từ đầu dữ liệu, rồi mới cắt cửa sổ SYNC) ----
stk = stock_long[stock_long.sku_id.isin(SKU_OK)].sort_values(KEY).copy()
stk['stock_delta'] = stk.groupby('sku_id')['stock_end'].diff()                      # tuần 26/06 làm mốc nên tuần SYNC_START vẫn có stock_delta
inb = im[im.MovementType.isin(['Opening Balance', 'Purchase Receipt']) & im.SKU_ID.isin(SKU_OK)]                  # giá vốn nhập kho (chỉ quá khứ)
uc = inb.assign(sku_id=inb.SKU_ID, week_start=week_start_of(inb.MovementDateTime)).groupby(KEY)['UnitCostAtMovement'].mean().rename('unit_cost').reset_index()
stk = stk.merge(uc, on=KEY, how='left'); stk['unit_cost'] = stk.groupby('sku_id')['unit_cost'].ffill()                # ffill: giá vốn nhập gần nhất ≤ t
es_stock_full = stk[(stk.week_start >= SYNC_START) & (stk.week_start <= SYNC_END)].drop(columns='delta_wk').reset_index(drop=True)
print(f'stock_end theo SKU–tuần trong cửa sổ SYNC: {len(es_stock_full):,} dòng = {es_stock_full.sku_id.nunique()} SKU × {es_stock_full.week_start.nunique()} tuần')
print('unit_cost thiếu:', int(es_stock_full.unit_cost.isna().sum()), '| stock_delta thiếu:', int(es_stock_full.stock_delta.isna().sum()))

In [ ]:
# ---- 2.2 nạp qty từ es_sales_wk (KHÔNG tính lại doanh số) và dựng lưới SKU–tuần đủ cửa sổ ----
df = es_stock_full.merge(es_sales_wk[KEY + ['qty']], on=KEY, how='left')
df['in_sales_range'] = df['qty'].notna(); df['qty'] = df['qty'].fillna(0)
df = df.sort_values(KEY).reset_index(drop=True)
g = df.groupby('sku_id', sort=False)['qty']

# ---- 2.3 cov_past, cov_fwd, n_fwd_weeks, trend4 ----
prev4 = g.transform(lambda x: x.shift(1).rolling(HORIZON_WEEKS, min_periods=HORIZON_WEEKS).mean())               # TB qty t-4…t-1
df['cov_past'] = df.stock_end / np.maximum(prev4, EPS)
fwd = pd.concat([g.shift(-k) for k in range(1, HORIZON_WEEKS + 1)], axis=1)                                      # qty t+1…t+4 (NaN nếu vượt SYNC_END)
df['n_fwd_weeks'] = fwd.notna().sum(axis=1).astype(int)
df['fwd_mean'] = fwd.mean(axis=1).where(df.n_fwd_weeks == HORIZON_WEEKS)                                         # chỉ khi đủ 4 tuần phía sau
df['cov_fwd'] = df.stock_end / np.maximum(df.fwd_mean, EPS)
l1_, l2_, l3_ = g.shift(1), g.shift(2), g.shift(3)
df['trend4'] = (1.5 * (df.qty - l3_) + 0.5 * (l1_ - l2_)) / 5                                                    # OLS slope t-3…t (≤ t)

# ---- kiểm tra định nghĩa trên một SKU và chống rò rỉ ----
s0 = df[df.sku_id == SKU_OK[0]].reset_index(drop=True); i0 = 50
assert abs(s0.cov_fwd[i0] - s0.stock_end[i0] / max(s0.qty[i0 + 1:i0 + 5].mean(), EPS)) < 1e-6 * max(1, abs(s0.cov_fwd[i0]))
assert abs(s0.cov_past[i0] - s0.stock_end[i0] / max(s0.qty[i0 - 4:i0].mean(), EPS)) < 1e-6 * max(1, abs(s0.cov_past[i0]))
assert df[df.n_fwd_weeks < HORIZON_WEEKS].cov_fwd.isna().all() and df[df.n_fwd_weeks == HORIZON_WEEKS].cov_fwd.notna().all()
assert df.groupby('sku_id').tail(HORIZON_WEEKS).n_fwd_weeks.max() < HORIZON_WEEKS
# đối chiếu trend4 với qty_slope4 của NB1 (cùng công thức, nơi NB1 có giá trị)
_cmp = df.merge(es_sales_wk[KEY + ['qty_slope4']], on=KEY).dropna(subset=['qty_slope4'])
print(f'Đối chiếu trend4 (NB2) vs qty_slope4 (NB1): {len(_cmp):,} dòng, chênh lệch tuyệt đối lớn nhất = {(_cmp.trend4 - _cmp.qty_slope4).abs().max():.2e}')
assert (_cmp.trend4 - _cmp.qty_slope4).abs().max() < 1e-9
print('✅ cov_past/cov_fwd/trend4 đúng định nghĩa; cov_fwd chỉ có khi đủ 4 tuần phía sau; trend4 chỉ dùng tuần ≤ t')
display(df[df.sku_id == SKU_OK[0]][KEY + ['qty', 'stock_end', 'stock_delta', 'cov_past', 'cov_fwd', 'trend4', 'n_fwd_weeks']].iloc[48:56].round(3))

### 2.4 Thống kê phân vị `cov_fwd` và số mẫu mất do thiếu 4 tuần phía sau
Mô phỏng đúng các bước 5.1–5.2 của notebook chung (nối `es_sales_wk` ⋈ `es_stock_wk`, đủ 4 tuần phía sau, bỏ dòng thiếu nhãn) để biết mẫu cuối cùng.

In [ ]:
EXPORT_COLS = ['sku_id', 'week_start', 'stock_end', 'stock_delta', 'cov_past', 'cov_fwd', 'trend4', 'n_fwd_weeks', 'unit_cost']
es_stock_wk = df[EXPORT_COLS].copy()
prev = es_sales_wk[KEY + ['y_qty']].merge(es_stock_wk, on=KEY, how='inner'); steps = [('es_sales_wk ⋈ es_stock_wk (inner)', len(prev))]
prev = prev[(prev.week_start >= SYNC_START) & (prev.week_start <= SYNC_END)]; steps.append(('trong cửa sổ SYNC', len(prev)))
prev = prev[prev.n_fwd_weeks >= HORIZON_WEEKS]; steps.append((f'đủ {HORIZON_WEEKS} tuần phía sau (n_fwd_weeks)', len(prev)))
prev = prev.dropna(subset=['y_qty', 'cov_fwd', 'stock_end']); prev = prev[np.isfinite(prev.cov_fwd)]; steps.append(('bỏ dòng thiếu nhãn (y_qty, cov_fwd)', len(prev)))
loss = pd.DataFrame(steps, columns=['bước', 'số_mẫu']); loss['mất_so_với_bước_trước'] = -loss.số_mẫu.diff().fillna(0).astype(int); display(loss)
n_lost_fwd = int((df.n_fwd_weeks < HORIZON_WEEKS).sum()); n_lost_sales = int(es_sales_wk.merge(es_stock_wk, on=KEY).pipe(lambda d: (d.n_fwd_weeks < HORIZON_WEEKS).sum()))
print(f'• Số SKU–tuần mất do thiếu 4 tuần phía sau: {n_lost_fwd:,} trên lưới đầy đủ ({len(SKU_OK)} SKU × {HORIZON_WEEKS} tuần cuối), trong đó {n_lost_sales:,} nằm trong khoảng bán của SKU.')
print(f'• Số SKU–tuần nằm ngoài khoảng bán đầu–cuối của SKU (không nối được với es_sales_wk): {int((~df.in_sales_range).sum()):,}')
fw0 = (df.loc[df.in_sales_range & (df.n_fwd_weeks == HORIZON_WEEKS), 'fwd_mean'] <= EPS).mean()
print(f'• Tỷ lệ mẫu có TB bán 4 tuần tới = 0 (cov_fwd bị chặn bởi ε): {fw0 * 100:.2f}%  → cov_fwd rất lớn (tồn kho không có cầu)')
pcts = [1, 5, 10, 25, 50, 55, 60, 65, 75, 80, 85, 90, 95, 99]
tbl_cov = pd.DataFrame({'cov_fwd (mẫu cuối cùng)': np.percentile(prev.cov_fwd, pcts), 'cov_past': np.nanpercentile(prev.cov_past, pcts)}, index=[f'P{p}' for p in pcts]); display(tbl_cov.round(3))
print('mean / std / max của cov_fwd:', round(prev.cov_fwd.mean(), 2), round(prev.cov_fwd.std(), 2), f'{prev.cov_fwd.max():.3g}')

### 2.5 Biểu đồ minh họa với **nhãn tạm** *(chỉ minh họa)*
> ⚠️ **Chỉ minh họa.** Ngưỡng tạm P60/P85 được tính trên **70% thời gian đầu** của mẫu; nhãn tạm `risk_tmp` **không được xuất** vào `es_stock_wk`. Ngưỡng chính thức chỉ tính trên `mt_train` ở notebook chung (Mục 5.6). Quy tắc tạm giống Mục 5.6: *High* = `cov_fwd ≥ T_high` **và** `trend4 ≤ 0`; *Medium* = `cov_fwd ≥ T_med` còn lại; còn lại *Low*.

In [ ]:
RISK_NAMES = {0: 'Low', 1: 'Medium', 2: 'High'}
wk_sorted = np.sort(prev.week_start.unique()); cut_w = wk_sorted[int(0.7 * len(wk_sorted)) - 1]
tmp_train = prev[prev.week_start <= cut_w]
T_MED, T_HIGH = np.percentile(tmp_train.cov_fwd, [60, 85])
def tmp_label(d, t_med, t_high):
    high = (d.cov_fwd >= t_high) & (d.trend4.fillna(0) <= 0); med = (d.cov_fwd >= t_med) & ~high
    return np.where(high, 2, np.where(med, 1, 0))
prev['risk_tmp'] = tmp_label(prev, T_MED, T_HIGH); prev['risk_tmp_name'] = prev.risk_tmp.map(RISK_NAMES)
print(f'Ngưỡng TẠM trên 70% thời gian đầu (đến tuần {pd.Timestamp(cut_w).date()}): T_med (P60) = {T_MED:.2f} tuần | T_high (P85) = {T_HIGH:.2f} tuần')
print('Tỷ lệ nhãn tạm (toàn mẫu):', prev.risk_tmp_name.value_counts(normalize=True).round(4).to_dict())
fig, ax = plt.subplots(figsize=(10, 4)); x = tmp_train.cov_fwd.clip(upper=tmp_train.cov_fwd.quantile(.99))
ax.hist(x, bins=70, color='#4C78A8'); ax.axvline(T_MED, c='orange', ls='--', label=f'P60 tạm = {T_MED:.1f}'); ax.axvline(T_HIGH, c='red', ls='--', label=f'P85 tạm = {T_HIGH:.1f}')
ax.set_title('Phân bố cov_fwd (70% thời gian đầu, cắt P99) – ngưỡng tạm\n(chỉ minh họa – ngưỡng chính thức tính trên train ở notebook chung)'); ax.set_xlabel('cov_fwd (số tuần bán còn lại)'); ax.legend(); plt.tight_layout(); plt.show(); save_fig(fig, 'hist_cov_fwd_nguong_tam')

In [ ]:
# ---- Hình 8: diễn biến các chỉ số theo nhãn tạm (theo tháng) ----
prev['month'] = prev.week_start.dt.to_period('M').dt.to_timestamp()
qty_now = df[KEY + ['qty']]; pv8 = prev.merge(qty_now, on=KEY, how='left')
colors = {'Low': '#54A24B', 'Medium': '#F58518', 'High': '#E45756'}
panels = [('stock_end', 'median', 'stock_end – trung vị'), ('qty', 'mean', 'qty tuần t – trung bình'), ('cov_past', 'median', 'cov_past – trung vị'), ('cov_fwd', 'median', 'cov_fwd – trung vị')]
fig, ax = plt.subplots(2, 2, figsize=(14, 7))
for a, (col, how, ttl) in zip(ax.ravel(), panels):
    for nm in ['Low', 'Medium', 'High']:
        s = pv8[pv8.risk_tmp_name == nm].groupby('month')[col].agg(how); a.plot(s.index, s.values, label=nm, c=colors[nm], lw=1.6)
    a.set_title(ttl); a.legend(fontsize=8)
    if col in ('cov_past', 'cov_fwd'): a.set_yscale('log')
fig.suptitle('Hình 8 – Diễn biến các chỉ số theo nhãn tạm (chỉ minh họa; ngưỡng tạm trên 70% thời gian đầu)', y=1.01); plt.tight_layout(); plt.show(); save_fig(fig, 'hinh8_dien_bien_chi_so_theo_nhan_tam')

In [ ]:
# ---- tỷ lệ High tạm theo tháng và theo nhóm sản phẩm ----
cat_map = es_sales_wk.drop_duplicates('sku_id').set_index('sku_id')['category']; prev['category'] = prev.sku_id.map(cat_map)
hm = prev.groupby('month')['risk_tmp'].apply(lambda s: (s == 2).mean()); hc = prev.groupby('category')['risk_tmp'].apply(lambda s: (s == 2).mean()).sort_values()
fig, ax = plt.subplots(1, 2, figsize=(15, 4))
ax[0].plot(hm.index, hm.values * 100, marker='o', ms=3, c='#E45756'); ax[0].axvline(pd.Timestamp(cut_w), c='gray', ls=':', label='mốc 70% (ngưỡng tạm)'); ax[0].set_ylabel('% High tạm'); ax[0].set_title('Tỷ lệ High tạm theo tháng (chỉ minh họa)'); ax[0].legend()
ax[1].barh(hc.index, hc.values * 100, color='#E45756'); ax[1].set_xlabel('% High tạm'); ax[1].set_title('Tỷ lệ High tạm theo nhóm sản phẩm (category)')
for i, v in enumerate(hc.values): ax[1].text(v * 100 + .2, i, f'{v * 100:.1f}%', va='center', fontsize=8)
plt.tight_layout(); plt.show(); save_fig(fig, 'ty_le_high_tam_theo_thang_va_nhom')

## 3. Kiểm chứng nhãn proxy *(sau Phần 2; chỉ để kiểm chứng, không dùng làm đặc trưng)*
`cov_fwd` là nhãn **proxy** của rủi ro tồn kho. Ba kiểm chứng: (a) phân bố nhãn tạm theo nhóm sản phẩm, (b) độ nhạy với ngưỡng, (c) mức giảm giá thực tế 4 tuần sau của nhóm High so với Low.
### 3.1 Phân bố nhãn tạm theo nhóm sản phẩm

In [ ]:
ct_n = pd.crosstab(prev.category, prev.risk_tmp_name)[['Low', 'Medium', 'High']]; ct_p = (pd.crosstab(prev.category, prev.risk_tmp_name, normalize='index')[['Low', 'Medium', 'High']] * 100).round(1)
display(ct_p.join(ct_n.sum(axis=1).rename('số_mẫu')).sort_values('High', ascending=False))
print(f"Tỷ lệ High giữa các nhóm: min = {ct_p.High.min():.1f}%, max = {ct_p.High.max():.1f}% (toàn mẫu {100 * (prev.risk_tmp == 2).mean():.1f}%)")
print('→ Nhãn có phân hóa theo nhóm sản phẩm (không dồn vào một nhóm duy nhất)' if ct_p.High.max() < 3 * max(ct_p.High.median(), 1) else '→ Một vài nhóm sản phẩm chiếm tỷ lệ High cao bất thường – xem xét đưa category vào đặc trưng/phân tầng')

### 3.2 Độ nhạy ngưỡng: `T_med` ∈ P55–P65, `T_high` ∈ P80–P90 (tính trên 70% thời gian đầu)

In [ ]:
rows = []
for pm in (55, 60, 65):
    for ph in (80, 85, 90):
        tm, th = np.percentile(tmp_train.cov_fwd, [pm, ph]); lab = tmp_label(prev, tm, th)
        rows.append({'T_med': f'P{pm}', 'T_high': f'P{ph}', 'T_med_giá_trị': round(tm, 2), 'T_high_giá_trị': round(th, 2), '%High': 100 * (lab == 2).mean(), '%Medium': 100 * (lab == 1).mean(), '%Low': 100 * (lab == 0).mean()})
sens = pd.DataFrame(rows); display(sens.round(2))
display(sens.pivot(index='T_med', columns='T_high', values='%High').round(2))
base_high = sens[(sens.T_med == 'P60') & (sens.T_high == 'P85')]['%High'].iloc[0]
print(f'%High: từ {sens["%High"].min():.2f}% đến {sens["%High"].max():.2f}% (biên độ {sens["%High"].max() - sens["%High"].min():.2f} điểm %) quanh cấu hình chuẩn P60/P85 = {base_high:.2f}%')
print(f'Thay đổi tương đối so với P60/P85: {((sens["%High"].min() / base_high) - 1) * 100:+.0f}% … {((sens["%High"].max() / base_high) - 1) * 100:+.0f}%')
print('Ghi chú: High cần cả cov_fwd ≥ T_high lẫn trend4 ≤ 0 nên %High nhỏ hơn (100 − P_high)%; so sánh với 12,2% ở kiểm chứng sơ bộ trong Thuyết minh.')

### 3.3 Mức giảm giá thực tế 4 tuần sau: High tạm vs Low
`ActualSellingPrice` đã được gộp ở NB1 thành `discount_rate` theo SKU–tuần. Mức giảm giá 4 tuần sau = trung bình `discount_rate` của tuần t+1…t+4 **có trọng số `qty`** (bỏ tuần không bán). Giả thuyết: nhóm High (tồn nhiều, cầu yếu) thường bị đẩy hàng bằng giảm giá nhiều hơn nhóm Low. **Chỉ để kiểm chứng.**

In [ ]:
sw = es_sales_wk.sort_values(KEY)[KEY + ['qty', 'discount_rate']].copy(); sw['wd'] = sw.qty * sw.discount_rate; gs = sw.groupby('sku_id', sort=False)
num = sum(gs['wd'].shift(-k) for k in range(1, HORIZON_WEEKS + 1)); den = sum(gs['qty'].shift(-k) for k in range(1, HORIZON_WEEKS + 1))
sw['disc_fwd4'] = (num / den.where(den > 0)); sw['disc_now'] = sw.discount_rate
vd = prev.merge(sw[KEY + ['disc_fwd4', 'disc_now']], on=KEY, how='left').dropna(subset=['disc_fwd4'])
tb = vd.groupby('risk_tmp_name').agg(số_mẫu=('disc_fwd4', 'size'), giảm_giá_4_tuần_sau_TB=('disc_fwd4', 'mean'), trung_vị=('disc_fwd4', 'median'), giảm_giá_tuần_t_TB=('disc_now', 'mean')).reindex(['Low', 'Medium', 'High'])
tb[['giảm_giá_4_tuần_sau_TB', 'trung_vị', 'giảm_giá_tuần_t_TB']] *= 100; display(tb.round(2))
hi, lo = vd.loc[vd.risk_tmp_name == 'High', 'disc_fwd4'], vd.loc[vd.risk_tmp_name == 'Low', 'disc_fwd4']
try:
    from scipy.stats import mannwhitneyu; p_mw = mannwhitneyu(hi, lo, alternative='two-sided').pvalue
except Exception: p_mw = np.nan
diff = (hi.mean() - lo.mean()) * 100
fig, ax = plt.subplots(figsize=(8, 4)); ax.boxplot([vd.loc[vd.risk_tmp_name == n, 'disc_fwd4'] * 100 for n in ['Low', 'Medium', 'High']], labels=['Low', 'Medium', 'High'], showfliers=False)
ax.set_ylabel('% giảm giá TB 4 tuần sau'); ax.set_title('Giảm giá thực tế 4 tuần sau theo nhãn tạm (chỉ kiểm chứng)'); plt.tight_layout(); plt.show(); save_fig(fig, 'kiem_chung_giam_gia_4_tuan_sau')
print(f'Chênh lệch giảm giá 4 tuần sau (High − Low) = {diff:+.2f} điểm % | Mann–Whitney p = {p_mw:.3g}')
print('→ Nhóm High tạm có mức giảm giá 4 tuần sau CAO HƠN Low có ý nghĩa thống kê: ủng hộ nhãn proxy.' if (diff > 0 and p_mw < 0.05) else
      '→ KHÔNG thấy giảm giá 4 tuần sau cao hơn có ý nghĩa ở nhóm High: nhãn proxy chưa được kiểm chứng bằng tín hiệu giảm giá – ghi nhận là hạn chế trong Thuyết minh (không dùng làm đặc trưng).')

## 4. Kiểm tra khóa → ghi file xuất
Với từng file: **khóa không trùng → số dòng → tuần min–max → số thiếu → mới ghi**. `es_stock_wk` **không** chứa nhãn lớp (`risk`, `risk_tmp`…).

In [ ]:
assert not {'risk', 'risk_name', 'risk_tmp', 'risk_tmp_name', 'in_sales_range', 'fwd_mean'} & set(es_stock_wk.columns), 'es_stock_wk không được chứa nhãn lớp'
REQ = ['sku_id', 'week_start', 'stock_end', 'cov_past', 'cov_fwd', 'n_fwd_weeks']; assert all(c in es_stock_wk.columns for c in REQ)
assert (es_stock_wk.stock_end >= 0).all(); assert es_stock_wk.sku_id.nunique() == len(SKU_OK)
es_quality_inventory = pd.DataFrame(QUALITY)
check_and_save(es_stock_wk,          'es_stock_wk',          key=KEY)
check_and_save(es_quality_inventory, 'es_quality_inventory')
print('File trong interim/:', sorted(p.name for p in DIRS['interim'].iterdir()))
display(es_stock_wk.describe().T.round(3))